# Ovine MOCAP visualizer

Visualization only. All processing logic lives in `simulation.mocap` (`io`, `kinematics`, `build`, `augment`) and `simulation.config.pipeline`.

To regenerate the data artifacts from the command line instead:
```
python -m simulation.mocap.run --augment
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from simulation.config import pipeline as P
from simulation.mocap import io, kinematics, build, augment

## 1. Load a C3D capture

In [ ]:
files = io.list_c3d_files()
for i, f in enumerate(files):
    print(f"[{i}] {f}")

FILE_INDEX = 0
c3d_path = P.C3D_DIR / files[FILE_INDEX]
io.describe_c3d(c3d_path)

df, point_rate = io.load_c3d(c3d_path)
df = kinematics.compute_joint_angles(df)
print("rows:", len(df), " point_rate:", point_rate, "Hz")

## 2. Skeleton animation (sagittal Y-Z plane)

In [ ]:
LEGS = {
    "Front R": ("R_GTUB", "R_LEPIRAD", "R_METAR", "RF_DPHAL", "red"),
    "Front L": ("L_GTUB", "L_LEPIRAD", "L_METAR", "LF_DPHAL", "blue"),
    "Back R":  ("R_GTROC", "R_LEPI", "R_LMAL", "RH_PPHAL", "green"),
    "Back L":  ("L_GTROC", "L_LEPI", "L_LMAL", "LH_PPHAL", "orange"),
}

fig, ax = plt.subplots()
ax.set_xlim(-750, 2500); ax.set_ylim(0, 800)
ax.set_xlabel("Y (horizontal)"); ax.set_ylabel("Z (vertical)")
scat = ax.scatter([], [], c="black", s=40, zorder=3)
lines = {name: ax.plot([], [], c=c, lw=2, label=name)[0]
         for name, (*_, c) in LEGS.items()}
time_text = ax.text(0.05, 0.9, "", transform=ax.transAxes)
ax.legend(loc="upper right")

def update(frame):
    offs = []
    for name, (a, b, c, d, _) in LEGS.items():
        ys = [df[f"{m}_Y"].iloc[frame] for m in (a, b, c, d)]
        zs = [df[f"{m}_Z"].iloc[frame] for m in (a, b, c, d)]
        lines[name].set_data(ys, zs)
        offs += list(zip(ys, zs))
    scat.set_offsets(offs)
    time_text.set_text(f"t = {df['Tiempo_s'].iloc[frame]:.2f} s")
    return (scat, *lines.values(), time_text)

ani = FuncAnimation(fig, update, frames=len(df), blit=True, interval=30)
P.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
ani.save(P.OUTPUTS_DIR / "esqueleto_movil.mp4", writer="ffmpeg", fps=30)
plt.close(fig)
print("saved", P.OUTPUTS_DIR / "esqueleto_movil.mp4")

## 3. Joint-angle kinematics

In [ ]:
def plot_legs(legs, title):
    fig, axes = plt.subplots(len(legs), 1, figsize=(12, 5 * len(legs)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, leg in zip(axes, legs):
        for joint in P.JOINT_ORDER:
            ax.plot(df["Tiempo_s"], df[f"{leg}_{joint}_angle"], lw=2, label=joint)
        ax.set_title(f"{title}: {leg}"); ax.set_ylabel("angle (deg)")
        ax.grid(True, ls="--", alpha=0.7); ax.legend(loc="upper right")
    axes[-1].set_xlabel("time (s)")
    plt.tight_layout(); plt.show()

plot_legs(["fl", "fr"], "Front legs")
plot_legs(["bl", "br"], "Back legs")

In [ ]:
kinematics.angle_summary(df)

## 4. Build trajectory and inspect gait cycles

In [ ]:
traj = build.angles_to_trajectory(df, frequency=point_rate)
print("qpos:", traj["qpos"].shape, " qvel:", traj["qvel"].shape)

troughs = augment.detect_cycles(traj["qpos"])
fr_hip = traj["qpos"][:, 7]
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(traj["time"], np.degrees(fr_hip), label="FR hip")
ax.scatter(traj["time"][troughs], np.degrees(fr_hip[troughs]),
           c="red", zorder=5, s=60, label=f"cycle bounds ({len(troughs)})")
ax.set_title("Gait cycle detection (FR hip troughs)")
ax.set_xlabel("time (s)"); ax.set_ylabel("angle (deg)")
ax.legend(); ax.grid(True, alpha=0.3); plt.show()

## 5. Augmentation sanity check

In [ ]:
aug = augment.generate_augmented(traj["qpos"], traj["qvel"], frequency=point_rate)
print("augmented cycles:", len(aug["meta"]), " qpos:", aug["qpos"].shape)

C = P.CYCLE_FRAMES
labels = [f"{leg.upper()} {j}" for leg in ["fr","fl","br","bl"] for j in P.JOINT_ORDER]
t_norm = np.linspace(0, 1, C)
fig, axes = plt.subplots(3, 4, figsize=(17, 9), sharex=True)
for k, (ax, label) in enumerate(zip(axes.flat, labels)):
    jidx = 7 + k
    for i, m in enumerate(aug["meta"]):
        if m["cycle"] != 0:
            continue
        seg = aug["qpos"][i * C:(i + 1) * C, jidx]
        color = "tomato" if "mirror" in m["type"] else "steelblue"
        ax.plot(t_norm, np.degrees(seg), alpha=0.2, lw=0.9, color=color)
    ax.set_title(label, fontsize=9); ax.grid(True, alpha=0.25)
fig.suptitle("Augmented variants, cycle 0 (blue=phase/speed/noise, red=mirror)")
plt.tight_layout(); plt.show()

# Persist if happy:
# build.save_trajectory(traj)
# augment.save_augmented(aug)